<a href="https://colab.research.google.com/github/SyedaMalaika75/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedaMalaika75/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*
### Finding 1 — 30-Day Momentum

The FlyRank report states that its 30-Day Momentum model predicts pages that will improve by more than 10% next month. It reports 95% performance on unseen pages from the same brands and 90% on brands the model has not seen before.

**My methodology question:**  
How was the “improve by more than 10% next month” label created, and were all predictor features restricted to information available before that future outcome window? I would also want to confirm that the unseen-brand evaluation was fully grouped by brand/client and compare the reported accuracy with the positive-class base rate.

### Finding 2 — Refreshing Pages Actually Works

The report says that 7 of 9 tested strata showed statistically significant impression lift after content refreshes.

**My methodology question:**  
How were refreshed pages selected and compared with non-refreshed pages? Could pages have been refreshed specifically because they were already declining or unusually weak? I would want to know how the analysis controlled for page age, prior traffic, regression to the mean, and other differences between refreshed and comparison pages before interpreting the observed lift as evidence of a refresh effect.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*
### Before vs after validation

My Week-5 model is Logistic Regression used as a ranking model.

For this audit, I compare a naive row-random split with the more honest client-grouped split.

**Before — random row split**
- Test base rate: 0.542
- Precision@20: 0.90
- Precision@50: 0.82

**After — grouped by client_id**
- Test base rate: 0.511
- Precision@20: 0.70
- Precision@50: 0.72
- Clients appearing in both train and test: 0

The random split gives more optimistic results because pages from the same client can appear in both training and testing. The grouped split is more conservative but better represents the question I care about: can the ranking generalize to clients the model did not see during training?

I therefore treat the grouped result as the honest evaluation result.

The grouped model still makes mistakes. In its top 50 ranked pages, 36 were observed decline cases and 14 were false positives. I also inspect actual decline cases ranked below the top 50 rather than relying only on the headline metric.

In [6]:
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "SyedaMalaika75/flyrank-ml-internship/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

# Observed target
df["decline_label"] = (
    df["trend_direction"] == "down"
).astype(int)


def prepare_frame(frame):
    frame = frame.copy()

    # avg_position = 0 means unavailable data, not rank zero
    frame["avg_position_missing"] = (
        frame["avg_position"] == 0
    ).astype(int)

    frame.loc[
        frame["avg_position"] == 0,
        "avg_position"
    ] = np.nan

    return frame


# Same safe feature set used in Week 5
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "ctr",
    "avg_position",
    "avg_position_missing",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
]

feature_columns = numeric_features + categorical_features


def make_model(num_features=numeric_features):

    numeric_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            ),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_transformer,
                num_features
            ),
            (
                "categorical",
                categorical_transformer,
                categorical_features
            ),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    random_state=42
                )
            ),
        ]
    )


def precision_at_k(labels, scores, k):
    result = pd.DataFrame({
        "label": np.asarray(labels),
        "score": scores
    })

    return (
        result
        .sort_values("score", ascending=False)
        .head(k)["label"]
        .mean()
    )


# --------------------------------------------------
# BEFORE: RANDOM ROW SPLIT
# --------------------------------------------------

random_train_idx, random_test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    random_state=42,
    stratify=df["decline_label"]
)

random_train = prepare_frame(
    df.iloc[random_train_idx]
)

random_test = prepare_frame(
    df.iloc[random_test_idx]
)

random_model = make_model()

random_model.fit(
    random_train[feature_columns],
    random_train["decline_label"]
)

random_scores = random_model.predict_proba(
    random_test[feature_columns]
)[:, 1]

random_p20 = precision_at_k(
    random_test["decline_label"],
    random_scores,
    20
)

random_p50 = precision_at_k(
    random_test["decline_label"],
    random_scores,
    50
)


# --------------------------------------------------
# AFTER: GROUPED CLIENT SPLIT
# --------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

group_train_idx, group_test_idx = next(
    splitter.split(
        df,
        y=df["decline_label"],
        groups=df["client_id"]
    )
)

group_train = prepare_frame(
    df.iloc[group_train_idx]
)

group_test = prepare_frame(
    df.iloc[group_test_idx]
)

group_model = make_model()

group_model.fit(
    group_train[feature_columns],
    group_train["decline_label"]
)

group_scores = group_model.predict_proba(
    group_test[feature_columns]
)[:, 1]

group_p20 = precision_at_k(
    group_test["decline_label"],
    group_scores,
    20
)

group_p50 = precision_at_k(
    group_test["decline_label"],
    group_scores,
    50
)

train_clients = set(group_train["client_id"])
test_clients = set(group_test["client_id"])
client_overlap = len(train_clients & test_clients)


comparison = pd.DataFrame({
    "Split": [
        "Random row split",
        "Grouped client split"
    ],
    "Test base rate": [
        random_test["decline_label"].mean(),
        group_test["decline_label"].mean()
    ],
    "Precision@20": [
        random_p20,
        group_p20
    ],
    "Precision@50": [
        random_p50,
        group_p50
    ]
})

display(comparison.round(3))

print("Grouped training rows:", len(group_train))
print("Grouped testing rows:", len(group_test))
print("Client overlap:", client_overlap)
print("Features used:", len(feature_columns))


# --------------------------------------------------
# REAL FAILURE EXAMPLES FROM GROUPED HOLDOUT
# --------------------------------------------------

errors = group_test[
    [
        "decline_label",
        "impressions_90d",
        "ctr",
        "avg_position",
        "days_since_last_update"
    ]
].copy()

errors["model_score"] = group_scores

errors = errors.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

errors["model_rank"] = errors.index + 1

top_50 = errors.head(50)

false_positives = top_50[
    top_50["decline_label"] == 0
].copy()

missed_declines = errors[
    (errors["decline_label"] == 1)
    & (errors["model_rank"] > 50)
].sort_values(
    "model_score",
    ascending=True
)

print(
    "\nTop-50 correct decline cases:",
    int(top_50["decline_label"].sum()),
    "/50"
)

print(
    "Top-50 false positives:",
    len(false_positives)
)

print("\nThree false-positive examples:")
display(
    false_positives[
        [
            "model_rank",
            "model_score",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update"
        ]
    ].head(3).round(3)
)

print("\nThree low-ranked observed decline examples:")
display(
    missed_declines[
        [
            "model_rank",
            "model_score",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update"
        ]
    ].head(3).round(3)
)


,Split,Test base rate,Precision@20,Precision@50
0,Random row split,0.542,0.9,0.82
1,Grouped client split,0.511,0.7,0.72


Grouped training rows: 23837
Grouped testing rows: 6163
Client overlap: 0
Features used: 32

Top-50 correct decline cases: 36 /50
Top-50 false positives: 14

Three false-positive examples:


,model_rank,model_score,impressions_90d,ctr,avg_position,days_since_last_update
4,5,0.901,235,0.85,31.0,20
6,7,0.894,3115,0.00,12.8,104
7,8,0.893,290,0.00,5.9,20



Three low-ranked observed decline examples:


,model_rank,model_score,impressions_90d,ctr,avg_position,days_since_last_update
6105,6106,0.018,1,0.00,NaN,92
6099,6100,0.085,916,0.00,78.6,22
6097,6098,0.088,83603,1.06,3.4,104


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*
### Leakage audit

My final model uses 32 input features.

The decline label is derived from `trend_direction`, which itself is based on `trend_pct`. Therefore both `trend_direction` and `trend_pct` are prohibited model inputs.

I also exclude IDs, provider/model names, and the recent 30-day comparison columns from the final feature set.

The automated overlap check finds no banned columns in the final feature list.

As an additional audit, I deliberately add `trend_pct` to a temporary test model. Because `trend_pct` is directly related to how the decline label is constructed, performance jumps to Precision@20 = 1.00 and Precision@50 = 1.00 on the grouped holdout. This is evidence that the leakage test is capable of detecting an answer-derived feature.

`trend_pct` is removed from the final model. I report only the leakage-free grouped result.

In [7]:
banned_features = {
    "content_id",
    "client_id",
    "provider_used",
    "model_used",
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
}

leaked_features = sorted(
    set(feature_columns) & banned_features
)

print(
    "Leaked features detected in final model:",
    leaked_features
)

assert not leaked_features


# Confirm the target really comes from trend_direction
label_check = (
    df["decline_label"]
    == (df["trend_direction"] == "down").astype(int)
).all()

print(
    "Label derivation check:",
    label_check
)


# --------------------------------------------------
# DELIBERATE LEAK TEST
# --------------------------------------------------

leaky_numeric_features = (
    numeric_features + ["trend_pct"]
)

leaky_feature_columns = (
    leaky_numeric_features
    + categorical_features
)

leaky_model = make_model(
    num_features=leaky_numeric_features
)

leaky_model.fit(
    group_train[leaky_feature_columns],
    group_train["decline_label"]
)

leaky_scores = leaky_model.predict_proba(
    group_test[leaky_feature_columns]
)[:, 1]

leaky_p20 = precision_at_k(
    group_test["decline_label"],
    leaky_scores,
    20
)

leaky_p50 = precision_at_k(
    group_test["decline_label"],
    leaky_scores,
    50
)

leak_test = pd.DataFrame({
    "Model": [
        "Final safe model",
        "Temporary model WITH trend_pct leak"
    ],
    "Precision@20": [
        group_p20,
        leaky_p20
    ],
    "Precision@50": [
        group_p50,
        leaky_p50
    ]
})

display(leak_test.round(3))

print(
    "\ntrend_pct present in FINAL feature set:",
    "trend_pct" in feature_columns
)

Leaked features detected in final model: []
Label derivation check: True


,Model,Precision@20,Precision@50
0,Final safe model,0.7,0.72
1,Temporary model WITH trend_pct leak,1.0,1.00



trend_pct present in FINAL feature set: False


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*
### Claim rewrite

**Earlier / too strong claim:**  
The model predicts which pages will decline.

**Rewritten public-safe claim:**  
On a client-grouped held-out evaluation, the Logistic Regression model ranked pages associated with observed decline patterns at Precision@20 of 0.70 and Precision@50 of 0.72, compared with a test base rate of approximately 0.511.

I treat this model as directional, decision-support evidence for deciding which pages may be worth reviewing first. The result does not prove that a page will decline, and it does not show that any individual feature causes decline.

The grouped result is more appropriate for my claim than the higher random-split score because no client appears in both training and testing.

In [8]:
claim_evidence = pd.DataFrame({
    "Evidence": [
        "Grouped test base rate",
        "Grouped Precision@20",
        "Grouped Precision@50",
        "Client overlap"
    ],
    "Value": [
        group_test["decline_label"].mean(),
        group_p20,
        group_p50,
        client_overlap
    ]
})

display(claim_evidence.round(3))

assert client_overlap == 0
assert not leaked_features

print(
    "Safe interpretation: directional decision-support, "
    "not a causal or guaranteed prediction."
)

,Evidence,Value
0,Grouped test base rate,0.511
1,Grouped Precision@20,0.700
2,Grouped Precision@50,0.720
3,Client overlap,0.000


Safe interpretation: directional decision-support, not a causal or guaranteed prediction.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.